# Imports

In [3]:
# importing different packages
import pandas as pd
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from shapely.geometry import Point
import requests
import time
import json
from io import StringIO

In [2]:
# Add utility territory and temperature data
df = pd.read_csv("../data/processed/combined_der_dataset_full.csv")
df["zip_code"] = df["zip_code"].astype(int).astype(str).str.zfill(5)
df = df[df["zip_code"].str.startswith("9")]
print(df["zip_code"])
print(sorted(df["zip_code"].unique()))

0       90001
1       90002
2       90003
3       90004
4       90005
        ...  
2581    98654
2582    98667
2583    99067
2584    99210
2585    99999
Name: zip_code, Length: 2579, dtype: object
['90001', '90002', '90003', '90004', '90005', '90006', '90007', '90008', '90009', '90010', '90011', '90012', '90013', '90014', '90015', '90016', '90017', '90018', '90019', '90020', '90021', '90022', '90023', '90024', '90025', '90026', '90027', '90028', '90029', '90030', '90031', '90032', '90033', '90034', '90035', '90036', '90037', '90038', '90039', '90040', '90041', '90042', '90043', '90044', '90045', '90046', '90047', '90048', '90049', '90050', '90051', '90053', '90054', '90055', '90056', '90057', '90058', '90059', '90060', '90061', '90062', '90063', '90064', '90065', '90066', '90067', '90068', '90069', '90071', '90073', '90074', '90075', '90077', '90078', '90079', '90080', '90082', '90083', '90086', '90089', '90091', '90093', '90094', '90095', '90201', '90204', '90209', '90210', '90211', 

# API Request to get ACS Data

In [3]:
# ACS (Newest 5-year) ZIP/ZCTA pull + clean predictors (race/poverty/education)
import requests
import pandas as pd
import numpy as np

api_key = "REMOVED_CENSUS_API_KEY"  # don't hardcode in public repos :)

YEAR = 2023
base_url = f"https://api.census.gov/data/{YEAR}/acs/acs5"

# --- Education (B15003: Educational Attainment for population 25+) ---
edu_vars = [
    "B15003_001E",  # total population 25+
    "B15003_017E",  # regular high school diploma
    "B15003_018E",  # GED/alternative
    "B15003_019E",  # some college < 1 year
    "B15003_020E",  # some college >= 1 year, no degree
    "B15003_021E",  # associate's
    "B15003_022E",  # bachelor's
    "B15003_023E",  # master's
    "B15003_024E",  # professional school
    "B15003_025E",  # doctorate
]

variables = [
    "B01003_001E",  # total population

    "B19013_001E",  # median household income
    "B25077_001E",  # median housing value

    # Race/ethnicity (Hispanic origin table)
    "B03002_001E",  # total
    "B03002_003E",  # White alone, not Hispanic or Latino
    "B03002_004E",  # Black alone, not Hispanic or Latino
    "B03002_006E",  # Asian alone, not Hispanic or Latino
    "B03002_012E",  # Hispanic or Latino (any race)

    # Poverty
    "B17001_001E",  # poverty universe
    "B17001_002E",  # below poverty level
] + edu_vars

# ---- Pull ZCTAs nationwide ----
params = {
    "get": ",".join(variables),
    "for": "zip code tabulation area:*",
    "key": api_key,
}
resp = requests.get(base_url, params=params, timeout=60)
resp.raise_for_status()

data = resp.json()
acs = pd.DataFrame(data[1:], columns=data[0])

# ---- Standardize ZIP in BOTH df and ACS before filtering ----
acs = acs.rename(columns={"zip code tabulation area": "zip_code"})
acs["zip_code"] = acs["zip_code"].astype(str).str.strip().str[:5].str.zfill(5)

df["zip_code"] = (
    df["zip_code"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)   # handle "94105.0"
    .str.replace(r"-.*$", "", regex=True)   # handle ZIP+4
    .str[:5]
    .str.zfill(5)
)

# ---- Convert numeric columns ----
for v in variables:
    acs[v] = pd.to_numeric(acs[v], errors="coerce")

# ---- Filter ACS to the ZIPs in your analysis panel ----
ca_zips = set(df["zip_code"])
acs = acs[acs["zip_code"].isin(ca_zips)].copy()

# ---- Rename columns nicely ----
rename_map = {
    "B01003_001E": "total_population",
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_housing_value",

    "B03002_001E": "raceeth_total",
    "B03002_003E": "white_not_hispanic",
    "B03002_004E": "black_not_hispanic",
    "B03002_006E": "asian_not_hispanic",
    "B03002_012E": "hispanic_any_race",

    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",

    "B15003_001E": "edu_25plus_total",
    "B15003_017E": "hs_diploma",
    "B15003_018E": "ged",
    "B15003_019E": "some_college_lt1yr",
    "B15003_020E": "some_college_ge1yr_no_degree",
    "B15003_021E": "associates",
    "B15003_022E": "bachelors",
    "B15003_023E": "masters",
    "B15003_024E": "professional_school",
    "B15003_025E": "doctorate",
}
acs = acs.rename(columns=rename_map)

# ---- Clean sentinel / invalid values (these drive your negative means) ----
# Medians should never be negative; negative values are missing/sentinel artifacts.
acs.loc[acs["median_household_income"] < 0, "median_household_income"] = pd.NA
acs.loc[acs["median_housing_value"] < 0, "median_housing_value"] = pd.NA

# Optional: drop ZCTAs with zero population (usually water/empty)
acs = acs[acs["total_population"] > 0].copy()

# ---- Derived shares/rates (safe denominators) ----
race_den = acs["raceeth_total"].replace({0: pd.NA})
pov_den  = acs["poverty_universe"].replace({0: pd.NA})
edu_den  = acs["edu_25plus_total"].replace({0: pd.NA})

acs["pct_black"]    = acs["black_not_hispanic"] / race_den
acs["pct_hispanic"] = acs["hispanic_any_race"] / race_den
acs["pct_asian"]    = acs["asian_not_hispanic"] / race_den
acs["poverty_rate"] = acs["below_poverty"] / pov_den

acs["pct_bachelors_plus"] = (
    acs[["bachelors", "masters", "professional_school", "doctorate"]].sum(axis=1) / edu_den
)

acs["pct_high_school_plus"] = (
    acs[[
        "hs_diploma", "ged",
        "some_college_lt1yr", "some_college_ge1yr_no_degree",
        "associates", "bachelors", "masters", "professional_school", "doctorate"
    ]].sum(axis=1) / edu_den
)

acs["pct_some_college_plus"] = (
    acs[[
        "some_college_lt1yr", "some_college_ge1yr_no_degree",
        "associates", "bachelors", "masters", "professional_school", "doctorate"
    ]].sum(axis=1) / edu_den
)

acs.drop(columns= ["raceeth_total"], inplace=True)

print("ACS rows after ZIP filter:", len(acs))
print("Unique ZIPs in df:", df["zip_code"].nunique())
print("Unique ZIPs matched in ACS:", acs["zip_code"].nunique())

# Quick sanity stats (after cleaning)
print(acs[["median_household_income", "median_housing_value"]].describe())
acs.head()

ACS rows after ZIP filter: 1752
Unique ZIPs in df: 2566
Unique ZIPs matched in ACS: 1752
       median_household_income  median_housing_value
count              1590.000000          1.594000e+03
mean              98418.279245          7.400354e+05
std               42746.886055          4.540993e+05
min               17663.000000          9.999000e+03
25%               67838.750000          3.916250e+05
50%               90051.000000          6.323500e+05
75%              119217.000000          9.507500e+05
max              250001.000000          2.000001e+06


,total_population,median_household_income,median_housing_value,white_not_hispanic,black_not_hispanic,asian_not_hispanic,hispanic_any_race,poverty_universe,below_poverty,edu_25plus_total,...,professional_school,doctorate,zip_code,pct_black,pct_hispanic,pct_asian,poverty_rate,pct_bachelors_plus,pct_high_school_plus,pct_some_college_plus
30595,56403,60751.0,513300.0,461,3871,263,51456,56189,11533,34437,...,139,54,90001,0.068631,0.912292,0.004663,0.205254,0.069954,0.49473,0.254203
30596,52735,56158.0,502600.0,213,7910,720,43219,52471,12265,31197,...,84,34,90002,0.149995,0.819551,0.013653,0.233748,0.080585,0.532038,0.289483
30597,71708,54781.0,547600.0,447,11669,258,58066,71189,18743,41791,...,20,17,90003,0.162729,0.809756,0.003598,0.263285,0.071379,0.534015,0.264961
30598,58844,62655.0,1457200.0,12237,2541,14755,27055,58617,10994,42864,...,1193,688,90004,0.043182,0.459775,0.250748,0.187557,0.400196,0.776596,0.60174
30599,38747,52755.0,1084400.0,4108,2072,12523,18498,38396,9463,28568,...,808,333,90005,0.053475,0.477405,0.323199,0.246458,0.379796,0.737258,0.531329


# Checking if the Missing Zips have Relevant Stats

In [4]:
model_predictors = [
    "median_household_income",
    "poverty_rate",
    "pct_bachelors_plus",
    "pct_black",
    "pct_hispanic",
    "pct_asian",
    # optional:
    "median_housing_value",
]
acs_model = acs[["zip_code"] + model_predictors + ["total_population"]].copy()
acs_model.to_csv(f'../data/processed/acs_predictors_ca_zip.csv', index=True)


matched = set(acs["zip_code"])
df["is_acs_matched"] = df["zip_code"].isin(matched).astype(int)
df.groupby("is_acs_matched")[["PV_system_size_DC","total_chargers","storage_capacity_mw"]].median()
df_model = df[df["zip_code"].isin(set(acs["zip_code"]))].copy()
df_model.drop(columns=["is_acs_matched"], inplace = True)
df_model.to_csv(f'../data/processed/combined_der_dataset_acs_matched.csv', index=True)

zip_code,ghi_mean_kwh_m2_day_2024,lat,lon
89410,5.04022301369863,38.78349434290129,-119.54760560489544
89020,5.504606849315069,36.562292913630664,-116.42458915970528
89444,5.04022301369863,38.726621208512874,-119.35076440322874

zip_code,wind_ws10m_mean_2024,wind_ws50m_mean_2024,lat,lon
89410,2.7164657534246577,3.715753424657536,38.78349434290129,-119.54760560489544
89020,3.7123561643835603,5.019424657534249,36.562292913630664,-116.42458915970528
89444,2.9159452054794506,3.9445205479452037,38.726621208512874,-119.35076440322874

# API Request to get Wind Speed Data

In [5]:
START = "20230101"
END   = "20231231"

# 1) Load ZCTAs
zcta = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/ZCTA520/tl_2023_us_zcta520.zip"
)
zcta["zip_code"] = zcta["ZCTA5CE20"].astype(str).str.zfill(5)

# 2) Load CA boundary and spatially filter ZCTAs to CA
states = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip"
)
ca = states.loc[states["STUSPS"] == "CA", ["geometry"]].to_crs(zcta.crs)

# keep ZCTAs that intersect CA at all (or use .within if you want stricter)
zcta_ca = gpd.sjoin(zcta, ca, predicate="intersects", how="inner").drop(columns=["index_right"])

# 3) Compute centroids the right way (planar CRS), then back to WGS84
zcta_ca = zcta_ca.to_crs("EPSG:3310")  # CA Albers (meters)
zcta_ca["centroid"] = zcta_ca.geometry.centroid
zcta_ca = zcta_ca.set_geometry("centroid").to_crs("EPSG:4326")
zcta_ca["lat"] = zcta_ca.geometry.y
zcta_ca["lon"] = zcta_ca.geometry.x

# 4) Fetch NASA POWER and store annual means per ZIP
session = requests.Session()

def fetch_wind_means(lat, lon, zip_code, sleep_s=0.2):
    """Return annual mean WS10M and WS50M for a single point."""
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=WS10M,WS50M&community=RE&longitude={lon}&latitude={lat}"
        f"&start={START}&end={END}&format=JSON"
    )
    r = session.get(url, timeout=60)
    r.raise_for_status()
    j = r.json()

    params = j.get("properties", {}).get("parameter", {})
    ws10 = params.get("WS10M", {})
    ws50 = params.get("WS50M", {})

    # Convert dict-of-days -> list of values, then mean
    ws10_vals = [v for v in ws10.values() if v is not None]
    ws50_vals = [v for v in ws50.values() if v is not None]

    out = {
        "zip_code": zip_code,
        "wind_ws10m_mean_2023": (sum(ws10_vals) / len(ws10_vals)) if ws10_vals else pd.NA,
        "wind_ws50m_mean_2023": (sum(ws50_vals) / len(ws50_vals)) if ws50_vals else pd.NA,
        "lat": lat,
        "lon": lon,
    }

    time.sleep(sleep_s)  # helps avoid rate limits
    return out

rows = []
for _, r in zcta_ca[["zip_code", "lat", "lon"]].drop_duplicates("zip_code").iterrows():
    try:
        rows.append(fetch_wind_means(r["lat"], r["lon"], r["zip_code"]))
    except Exception as e:
        print(f"Failed ZIP {r['zip_code']}: {e}")

df_wind = pd.DataFrame(rows)
df_wind.to_csv("../data/processed/ca_zip_wind_means_2023.csv", index=False)
print("Saved:", df_wind.shape)

Saved: (1846, 5)


# API Request to get Solar Irradiance Data

In [6]:
# ---- Date range (NASA POWER wants YYYYMMDD) ----
START = "20230101"
END   = "20231231"

# 1) Load ZCTA boundaries
zcta = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/ZCTA520/tl_2023_us_zcta520.zip"
)
zcta["zip_code"] = zcta["ZCTA5CE20"].astype(str).str.zfill(5)

# 2) Load CA boundary + spatially filter ZCTAs to California
states = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip"
)
ca = states.loc[states["STUSPS"] == "CA", ["geometry"]].to_crs(zcta.crs)

# Keep ZCTAs that intersect CA
zcta_ca = gpd.sjoin(zcta, ca, predicate="intersects", how="inner").drop(columns=["index_right"])

# 3) Compute centroids correctly (project first), then convert back to lat/lon
zcta_ca = zcta_ca.to_crs("EPSG:3310")  # CA Albers (meters)
zcta_ca["centroid"] = zcta_ca.geometry.centroid
zcta_ca = zcta_ca.set_geometry("centroid").to_crs("EPSG:4326")

zcta_ca["lat"] = zcta_ca.geometry.y
zcta_ca["lon"] = zcta_ca.geometry.x

# 4) NASA POWER fetch: annual mean GHI (ALLSKY_SFC_SW_DWN, kWh/m^2/day)
session = requests.Session()

def fetch_ghi_mean(lat, lon, zip_code, sleep_s=0.2):
    """Return annual mean daily GHI for 2023 at a point (kWh/m^2/day)."""
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=ALLSKY_SFC_SW_DWN&community=RE&longitude={lon}&latitude={lat}"
        f"&start={START}&end={END}&format=JSON"
    )
    r = session.get(url, timeout=60)
    r.raise_for_status()
    j = r.json()

    params = j.get("properties", {}).get("parameter", {})
    ghi = params.get("ALLSKY_SFC_SW_DWN", {})

    vals = [v for v in ghi.values() if v is not None]
    out = {
        "zip_code": zip_code,
        "ghi_mean_kwh_m2_day_2023": (sum(vals) / len(vals)) if vals else pd.NA,
        "lat": lat,
        "lon": lon,
    }

    time.sleep(sleep_s)
    return out

rows = []
for _, r in zcta_ca[["zip_code", "lat", "lon"]].drop_duplicates("zip_code").iterrows():
    try:
        rows.append(fetch_ghi_mean(r["lat"], r["lon"], r["zip_code"]))
    except Exception as e:
        print(f"Failed ZIP {r['zip_code']}: {e}")

df_solar = pd.DataFrame(rows)
df_solar.to_csv("../data/processed/ca_zip_ghi_mean_2023.csv", index=False)
print("✅ Saved:", df_solar.shape, "to ../data/processed/ca_zip_ghi_mean_2023.csv")

✅ Saved: (1846, 4) to ../data/processed/ca_zip_ghi_mean_2023.csv


# API Request to get Temperature Data

In [7]:
# ----------------------------
# Config
# ----------------------------
YEAR = 2023
START = f"{YEAR}0101"
END   = f"{YEAR}1231"

OUT_CSV = f"../data/processed/ca_zip_temperature_controls_{YEAR}.csv"

# NASA POWER temperature params (°C)
TEMP_PARAMS = ["T2M", "T2M_MAX", "T2M_MIN"]  # daily mean/max/min at 2m

# Degree-day base (F)
BASE_F = 65.0

# Rate-limit sleep (seconds) to be nice to the API
SLEEP_S = 0.2


# ----------------------------
# 1) ZCTAs -> California only
# ----------------------------
zcta = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/ZCTA520/tl_2023_us_zcta520.zip"
)
zcta["zip_code"] = zcta["ZCTA5CE20"].astype(str).str.zfill(5)

states = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip"
)
ca = states.loc[states["STUSPS"] == "CA", ["geometry"]].to_crs(zcta.crs)

# Keep ZCTAs that intersect California at all
zcta_ca = gpd.sjoin(zcta, ca, predicate="intersects", how="inner").drop(columns=["index_right"])

# Compute centroids correctly (projected CRS -> WGS84 lat/lon)
zcta_ca = zcta_ca.to_crs("EPSG:3310")  # CA Albers
zcta_ca["centroid"] = zcta_ca.geometry.centroid
zcta_ca = zcta_ca.set_geometry("centroid").to_crs("EPSG:4326")

zcta_ca["lat"] = zcta_ca.geometry.y
zcta_ca["lon"] = zcta_ca.geometry.x

zcta_points = zcta_ca[["zip_code", "lat", "lon"]].drop_duplicates("zip_code").copy()


# ----------------------------
# 2) NASA POWER fetch helpers
# ----------------------------
session = requests.Session()

def c_to_f(c):
    return (c * 9.0 / 5.0) + 32.0

def fetch_daily_temp_series(lat, lon):
    """
    Fetch daily temp series from NASA POWER for a single point.
    Returns dicts: {"T2M": {YYYYMMDD: val, ...}, ...}
    Values are in °C.
    """
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters={','.join(TEMP_PARAMS)}"
        "&community=RE"
        f"&longitude={lon}&latitude={lat}"
        f"&start={START}&end={END}"
        "&format=JSON"
    )
    r = session.get(url, timeout=60)
    r.raise_for_status()
    j = r.json()
    params = j.get("properties", {}).get("parameter", {})
    return {p: params.get(p, {}) for p in TEMP_PARAMS}

def summarize_temp_controls(zip_code, lat, lon, sleep_s=SLEEP_S):
    """
    For one ZIP (centroid point), compute:
      - annual means of T2M/T2M_MAX/T2M_MIN (°C)
      - summer (Jun–Sep) mean T2M (°C)
      - HDD65 and CDD65 using daily mean temp (°F)
    """
    series = fetch_daily_temp_series(lat, lon)

    t2m = series["T2M"]
    tmax = series["T2M_MAX"]
    tmin = series["T2M_MIN"]

    # Convert dict values to lists (skip None)
    t2m_vals  = [v for v in t2m.values()  if v is not None]
    tmax_vals = [v for v in tmax.values() if v is not None]
    tmin_vals = [v for v in tmin.values() if v is not None]

    # Summer mean (Jun–Sep)
    summer_vals = []
    for d, v in t2m.items():
        if v is None:
            continue
        # d is "YYYYMMDD"
        month = int(d[4:6])
        if 6 <= month <= 9:
            summer_vals.append(v)

    # Degree days (using daily mean T2M)
    cdd = 0.0
    hdd = 0.0
    for v in t2m_vals:
        tf = c_to_f(v)
        cdd += max(0.0, tf - BASE_F)
        hdd += max(0.0, BASE_F - tf)

    out = {
        "zip_code": zip_code,
        "lat": lat,
        "lon": lon,

        # Annual means (°C)
        f"t2m_mean_c_{YEAR}": (sum(t2m_vals) / len(t2m_vals)) if t2m_vals else pd.NA,
        f"t2m_max_mean_c_{YEAR}": (sum(tmax_vals) / len(tmax_vals)) if tmax_vals else pd.NA,
        f"t2m_min_mean_c_{YEAR}": (sum(tmin_vals) / len(tmin_vals)) if tmin_vals else pd.NA,

        # Seasonal mean (°C)
        f"t2m_summer_mean_c_{YEAR}": (sum(summer_vals) / len(summer_vals)) if summer_vals else pd.NA,

        # Degree-day totals (°F base)
        f"cdd65_{YEAR}": cdd if t2m_vals else pd.NA,
        f"hdd65_{YEAR}": hdd if t2m_vals else pd.NA,
    }

    time.sleep(sleep_s)
    return out


# ----------------------------
# 3) Loop + save
# ----------------------------
rows = []
for _, r in zcta_points.iterrows():
    z = r["zip_code"]
    try:
        rows.append(summarize_temp_controls(z, r["lat"], r["lon"]))
    except Exception as e:
        print(f"Failed ZIP {z}: {e}")

df_temp = pd.DataFrame(rows)
df_temp.to_csv(OUT_CSV, index=False)

print("✅ Saved temperature controls:", df_temp.shape, "->", OUT_CSV)
df_temp.head()

✅ Saved temperature controls: (1846, 9) -> ../data/processed/ca_zip_temperature_controls_2023.csv


,zip_code,lat,lon,t2m_mean_c_2023,t2m_max_mean_c_2023,t2m_min_mean_c_2023,t2m_summer_mean_c_2023,cdd65_2023,hdd65_2023
0,89410,38.783494,-119.547606,10.347096,17.540740,4.499041,20.699262,732.084,5979.042
1,89020,36.562293,-116.424589,17.504192,24.718356,10.755233,27.873934,2532.408,3077.154
2,89444,38.726621,-119.350764,6.832082,13.939699,0.999068,16.707213,226.650,7782.972
3,89048,36.164546,-115.988332,18.895151,26.244986,11.976055,29.016148,2922.780,2553.666
4,89061,36.071278,-115.899461,15.921288,23.009836,9.478000,26.192377,2024.490,3609.204


# Utility Territory
https://www.arcgis.com/home/item.html?id=0e0e2c9cc0e04d9e86e323cc26bbf94d

In [8]:
# -----------------------------
# Inputs
# -----------------------------
SCOUT_FEATURESERVER = "https://services.arcgis.com/BLN4oKB0N1YSgvY8/arcgis/rest/services/California_Electric_Utility_Service_Territory_SCOUT/FeatureServer"
SCOUT_LAYER_ID = 0  # "Electric Service Areas"

ZCTA_URL  = "https://www2.census.gov/geo/tiger/TIGER2023/ZCTA520/tl_2023_us_zcta520.zip"
STATE_URL = "https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip"

OUT_CSV = "../data/processed/zip_to_utility.csv"


# -----------------------------
# Helpers
# -----------------------------
def fetch_arcgis_layer_as_gdf(
    feature_server_url: str,
    layer_id: int = 0,
    where: str = "1=1",
    out_fields: str = "*",
    out_sr: int = 4326,
    page_size: int = 2000,
) -> gpd.GeoDataFrame:
    """
    Download ALL features (including geometry) from an ArcGIS FeatureServer layer
    into a GeoDataFrame using pagination.

    Notes
    -----
    - ArcGIS services often enforce maxRecordCount (commonly 2000).
    - We request GeoJSON output so GeoPandas can ingest directly.
    """
    query_url = f"{feature_server_url}/{layer_id}/query"

    # Count first
    r = requests.get(
        query_url,
        params={"where": where, "returnCountOnly": "true", "f": "json"},
        timeout=60,
    )
    r.raise_for_status()
    total = int(r.json().get("count", 0))

    chunks = []
    offset = 0

    while offset < total:
        params = {
            "where": where,
            "outFields": out_fields,
            "returnGeometry": "true",
            "outSR": out_sr,
            "f": "geojson",
            "resultRecordCount": page_size,
            "resultOffset": offset,
        }
        resp = requests.get(query_url, params=params, timeout=120)
        resp.raise_for_status()
        gj = resp.json()

        # Defensive: some servers return {"error": ...}
        if "features" not in gj:
            raise RuntimeError(f"Unexpected response (no 'features'): {gj}")

        gdf_chunk = gpd.GeoDataFrame.from_features(gj["features"], crs=f"EPSG:{out_sr}")
        if gdf_chunk.empty:
            break

        chunks.append(gdf_chunk)
        offset += len(gdf_chunk)

    if not chunks:
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{out_sr}")

    return pd.concat(chunks, ignore_index=True)


def load_california_boundary() -> gpd.GeoDataFrame:
    """
    Load the US state polygons and return California as a single dissolved polygon.

    Why dissolve?
    - Some state layers can include multipart geometries; dissolve ensures we have
      one CA geometry for clean clipping.
    """
    states = gpd.read_file(STATE_URL).to_crs(epsg=4326)
    ca = states[states["STUSPS"] == "CA"]
    return ca.dissolve()


def build_zip_to_scout_utility() -> pd.DataFrame:
    """
    Build a ZIP->utility crosswalk using ONLY the SCOUT utility service territory layer.

    Steps
    -----
    1) Load SCOUT polygons (service territories).
    2) Load ZCTAs and clip them to CA boundary.
    3) Intersect ZIP polygons with territory polygons.
    4) Compute overlap area (m^2) in CA Albers (EPSG:3310).
    5) Assign each ZIP to the utility with the largest overlap.

    Output columns
    --------------
    zip_code, acronym, utility, overlap_area_m2, overlap_share_of_zip
    """
    # 1) Load SCOUT territories
    territories_raw = fetch_arcgis_layer_as_gdf(
        SCOUT_FEATURESERVER,
        layer_id=SCOUT_LAYER_ID,
        out_fields="Acronym,Utility,Type",  # keep it lean; add more if you want
        out_sr=4326,
    )
    territories_raw = territories_raw[~territories_raw.geometry.isna()].copy()

    # Normalize names
    territories = territories_raw.rename(
        columns={"Acronym": "acronym", "Utility": "utility", "Type": "utility_type"}
    )

    # Sanity check (helps catch definitionQuery limitations)
    print("SCOUT utilities fetched:\n", territories["utility"].value_counts(dropna=False).head(20))

    # 2) Load ZCTAs and clip to CA
    zcta = gpd.read_file(ZCTA_URL).to_crs(epsg=4326)
    zcta = zcta.rename(columns={"ZCTA5CE20": "zip_code"})
    zcta["zip_code"] = zcta["zip_code"].astype(str).str.strip().str[:5].str.zfill(5)

    ca = load_california_boundary()
    zcta_ca = gpd.overlay(zcta[["zip_code", "geometry"]], ca, how="intersection")

    # 3) Area-safe intersection in equal-area CRS
    territories_aea = territories.to_crs(epsg=3310)
    zcta_aea = zcta_ca.to_crs(epsg=3310)

    inter = gpd.overlay(zcta_aea, territories_aea, how="intersection")
    if inter.empty:
        raise RuntimeError("No intersections found. Check CRS and that territories cover CA.")

    inter["overlap_area_m2"] = inter.geometry.area

    # ZIP areas for overlap share
    zip_area = zcta_aea.copy()
    zip_area["zip_area_m2"] = zip_area.geometry.area
    inter = inter.merge(zip_area[["zip_code", "zip_area_m2"]], on="zip_code", how="left")
    inter["overlap_share_of_zip"] = inter["overlap_area_m2"] / inter["zip_area_m2"]

    # 4) Dominant utility per ZIP
    idx = inter.groupby("zip_code")["overlap_area_m2"].idxmax()
    dom = inter.loc[idx, ["zip_code", "acronym", "utility", "utility_type", "overlap_area_m2", "overlap_share_of_zip"]].copy()
    dom = dom.sort_values("zip_code").reset_index(drop=True)

    return dom


# -----------------------------
# Run + write
# -----------------------------
zip_to_utility_scout = build_zip_to_scout_utility()
zip_to_utility_scout.to_csv(OUT_CSV, index=False)

print("Wrote:", OUT_CSV)
print("ZIPs mapped:", zip_to_utility_scout["zip_code"].nunique())
print(zip_to_utility_scout.head())

SCOUT utilities fetched:
 utility
Southern California Edison        1
Pacific Gas & Electric Company    1
San Diego Gas & Electric          1
Name: count, dtype: int64


/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_21713/1523109546.py:130: UserWarning: `keep_geom_type=True` in overlay resulted in 43 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  zcta_ca = gpd.overlay(zcta[["zip_code", "geometry"]], ca, how="intersection")


Wrote: ../data/processed/zip_to_utility.csv
ZIPs mapped: 1660
  zip_code acronym                     utility utility_type  overlap_area_m2  \
0    89019     SCE  Southern California Edison          IOU     2.901458e+08   
1    89060     SCE  Southern California Edison          IOU     2.196799e+07   
2    89061     SCE  Southern California Edison          IOU     7.370155e+06   
3    90001     SCE  Southern California Edison          IOU     6.452122e+06   
4    90002     SCE  Southern California Edison          IOU     2.629105e+06   

   overlap_share_of_zip  
0              1.000000  
1              0.999889  
2              1.000000  
3              0.759482  
4              0.339210  


Check if there is a dataset that describes the type of area (residential or industrial) to explain the positive coefficient of poverty rate

In [9]:

pge = None

for q in [1, 2, 3, 4]:
    pge_temp = pd.read_csv(f"../data/raw/demand/PGE_2023_Q{q}_ElectricUsageByZip.csv")
    pge_temp.columns = pge_temp.columns.str.strip().str.lower()
    pge_temp = pge_temp.drop(columns=["month", "year", "customerclass", "combined"], errors="ignore")
    pge_temp = pge_temp.rename(columns={"zipcode": "zip_code"})

    pge_temp["zip_code"] = pge_temp["zip_code"].astype(int).astype(str).str.zfill(5)

    for col in ["totalcustomers", "totalkwh", "averagekwh"]:
        if col in pge_temp.columns:
            pge_temp[col] = pd.to_numeric(pge_temp[col], errors="coerce")

    pge_temp = pge_temp.groupby("zip_code").sum(numeric_only=True)
    pge_temp = pge_temp.add_suffix(f"_q{q}")

    if pge is None:
        pge = pge_temp
    else:
        pge = pge.join(pge_temp, how="outer")  # join on index (zip_code)

sce = None

for q in [1, 2, 3, 4]:
    sce_temp = pd.read_excel(
        f"../data/raw/demand/SCE_2023_Q{q}_ElectricUsageByZip.xlsx",
        skiprows=[0, 1]
    )

    sce_temp.columns = sce_temp.columns.str.strip().str.lower()
    sce_temp = sce_temp.drop(columns=["month", "year", "customer\nclass", "combined"], errors="ignore")
    sce_temp = sce_temp.rename(columns={"zip\ncode": "zip_code"})

    sce_temp["zip_code"] = sce_temp["zip_code"].astype(int).astype(str).str.zfill(5)
    for col in ["totalaccounts", "totalkwh", "averagekwh"]:
        if col in sce_temp.columns:
            sce_temp[col] = pd.to_numeric(sce_temp[col], errors="coerce")

    sce_temp = sce_temp.groupby("zip_code").sum(numeric_only=True)
    sce_temp = sce_temp.add_suffix(f"_q{q}")

    if sce is None:
        sce = sce_temp
    else:
        sce = sce.join(sce_temp, how="outer")

sdge = None

for q in [1, 2, 3, 4]:
    sdge_temp = pd.read_csv(
        f"../data/raw/demand/SDGE-ELEC-2023-Q{q}.csv",
    )

    sdge_temp.columns = sdge_temp.columns.str.strip().str.lower()
    sdge_temp = sdge_temp.drop(columns=["month", "year", "customerclass", "combined"], errors="ignore")
    sdge_temp = sdge_temp.rename(columns={"zipcode": "zip_code"})
    sdge_temp["zip_code"] = sdge_temp["zip_code"].astype(int).astype(str).str.zfill(5)
    for col in ["totalaccounts", "totalkwh", "averagekwh"]:
        if col in sdge_temp.columns:
            sdge_temp[col] = pd.to_numeric(sdge_temp[col], errors="coerce")

    sdge_temp = sdge_temp.groupby("zip_code").sum(numeric_only=True)
    sdge_temp = sdge_temp.add_suffix(f"_q{q}")

    if sdge is None:
        sdge = sdge_temp
    else:
        sdge = sdge.join(sdge_temp, how="outer")


/opt/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/opt/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/opt/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/opt/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [10]:
# -----------------------
# helper: per-utility annual aggregation
# -----------------------
def annualize_utility(df, util, cust_col_base):
    """
    df: wide quarterly df indexed by zip_code with columns like totalkwh_q1..q4 and <cust_col_base>_q1..q4
    util: string prefix like "pge"
    cust_col_base: "totalcustomers" or "totalaccounts"
    """
    out = pd.DataFrame(index=df.index)

    # annual kWh = sum quarters
    kwh_cols = [f"totalkwh_q{i}" for i in range(1, 5)]
    out[f"{util}_kwh_annual"] = df.reindex(columns=kwh_cols).sum(axis=1, min_count=1)

    # customer-months ~= 3 * sum(quarterly customer counts)
    cust_cols = [f"{cust_col_base}_q{i}" for i in range(1, 5)]
    cust_q_sum = df.reindex(columns=cust_cols).sum(axis=1, min_count=1)
    out[f"{util}_cust_months"] = 3 * cust_q_sum

    # kWh per customer-month (avoid divide-by-zero)
    out[f"{util}_kwh_per_cust_month"] = out[f"{util}_kwh_annual"] / out[f"{util}_cust_months"].replace(0, np.nan)

    return out


# -----------------------
# 1) annual per-utility tables
# -----------------------
pge_a  = annualize_utility(pge,  "pge",  cust_col_base="totalcustomers")
sce_a  = annualize_utility(sce,  "sce",  cust_col_base="totalaccounts")
sdge_a = annualize_utility(sdge, "sdge", cust_col_base="totalaccounts")

# join all utilities into one ZIP-level demand table
demand = pge_a.join(sce_a, how="outer").join(sdge_a, how="outer")

# -----------------------
# 2) aggregate across utilities to ZIP totals (+ utility structure features)
# -----------------------
kwh_cols   = [c for c in demand.columns if c.endswith("_kwh_annual")]
custm_cols = [c for c in demand.columns if c.endswith("_cust_months")]

demand["kwh_annual_total"] = demand[kwh_cols].sum(axis=1, min_count=1)
demand["cust_months_total"] = demand[custm_cols].sum(axis=1, min_count=1)
demand["kwh_per_cust_month_total"] = demand["kwh_annual_total"] / demand["cust_months_total"].replace(0, np.nan)

# how many utilities contribute data in that ZIP
demand["num_utils_reporting"] = demand[kwh_cols].gt(0).sum(axis=1)

# share of the dominant utility (flags split ZIPs)
demand["dominant_util_share"] = demand[kwh_cols].max(axis=1) / demand["kwh_annual_total"].replace(0, np.nan)
demand = demand.reset_index()
demand_control = pd.DataFrame()
demand_control["zip_code"] = demand["zip_code"]
demand_control["kwh_annual_total"] = demand["kwh_annual_total"]
demand_control["log_kwh"] = np.log1p(demand_control["kwh_annual_total"])
demand_control.to_csv("../data/processed/demand.csv")

In [10]:
# Energy burden / affordability ZIP-level features
from pathlib import Path
ENERGY_BURDEN_DIR = Path("../data/raw/energy_burden")

ENERGY_BURDEN_SOURCES = {
    "energy_burden_pct": ("EB_data.csv", "EB.2"),
    "energy_affordability_index": ("EAI_data.csv", "EA.7"),
    "energy_affordability_gap": ("EA Gap_data.csv", "EA Gap.3"),
}

def _parse_energy_burden_value(series, as_percent=False):
    values = (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan})
    )
    values = pd.to_numeric(values, errors="coerce")
    if as_percent:
        values = values / 100
    return values

def load_energy_burden_zip_data(data_dir=ENERGY_BURDEN_DIR):
    """Load CA ZIP-level energy burden metrics from UTF-16 tab-delimited Tableau exports."""
    parts = []
    for out_col, (filename, source_col) in ENERGY_BURDEN_SOURCES.items():
        raw = pd.read_csv(data_dir / filename, encoding="utf-16", sep="\t")
        zips = raw["ZIP_CODE"].astype(str).str.extract(r"(\d{5})", expand=False).str.zfill(5)
        values = _parse_energy_burden_value(raw[source_col], as_percent=(out_col == "energy_burden_pct"))
        parts.append(pd.DataFrame({"zip_code": zips, out_col: values}))

    energy = parts[0]
    for part in parts[1:]:
        energy = energy.merge(part, on="zip_code", how="outer")

    energy = energy.drop_duplicates(subset="zip_code").copy()
    return energy

energy_burden = load_energy_burden_zip_data()
energy_burden.to_csv(f'../data/processed/energy_burden.csv', index=True)